<a href="https://colab.research.google.com/github/dipakphp/Customer_Retention_Analytics/blob/main/Customer_Retention_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Engagement & Product Utilization Analytics for Retention Strategy



### Project Objectives:
1. **Evaluate Engagement vs Churn**
2. **Measure Product Utilization**
3. **Detect High-Value Disengaged Customers**
4. **Score Relationship Strength**
5. **Expose Streamlit Dashboard**

## Step 1: Environment Setup and Library Installation
Installing the libraries required for analytics, visualization, and dashboard deployment.

In [1]:
# Install required packages
!pip install -q pandas numpy plotly streamlit scikit-learn

# Install localtunnel to expose the Streamlit app from Colab
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 95.7 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 3s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹

## Step 2: Data Ingestion & Validation
We load the `European_Bank.csv` dataset.

In [2]:
import pandas as pd
import numpy as np
import os
import urllib.request
from google.colab import files

file_name = "European_Bank.csv"

if not os.path.exists(file_name):
    print("Please upload your 'European_Bank.csv' file:")
    uploaded = files.upload()
    # If nothing was uploaded, download base dataset as fallback
    if not uploaded:
        print("No file uploaded. Downloading base dataset from GitHub...")
        url = "https://raw.githubusercontent.com/tushar-rishav/Bank-Customer-Churn-Prediction/master/Churn_Modelling.csv"
        urllib.request.urlretrieve(url, file_name)

        # Load and append 'Year' column to match screenshot
        df_base = pd.read_csv(file_name)
        # Add Year column (randomly assign 2025 or 2026)
        np.random.seed(42)
        df_base.insert(0, 'Year', np.random.choice([2025, 2026], size=len(df_base)))
        df_base.to_csv(file_name, index=False)
        print("Downloaded and prepared European_Bank.csv successfully!")

# Load and validate data
df = pd.read_csv(file_name)
print(f"Dataset Loaded successfully. Shape: {df.shape}")
df.head()

Dataset Loaded successfully. Shape: (10000, 14)


,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Step 3: Analytical Methodology & Calculations

We define our analytical functions here. This includes engagement profile classification, KPI calculations, relationship scoring, and premium customer detection.

In [3]:
def classify_engagement(df, balance_threshold=100000.0):
    df = df.copy()
    conditions = [
        (df["IsActiveMember"] == 1) & (df["NumOfProducts"] >= 2),
        (df["IsActiveMember"] == 1) & (df["NumOfProducts"] == 1),
        (df["IsActiveMember"] == 0) & (df["Balance"] >= balance_threshold),
        (df["IsActiveMember"] == 0) & (df["Balance"] < balance_threshold)
    ]
    choices = [
        "Active Engaged",
        "Active Low-Product",
        "Inactive High-Balance",
        "Inactive Disengaged"
    ]
    df["EngagementProfile"] = np.select(conditions, choices, default="Inactive Disengaged")
    return df

def calculate_rsi(df):
    df = df.copy()
    # Active points (40 pts)
    active_pts = df["IsActiveMember"] * 40

    # Product depth points (30 pts)
    prod_conditions = [
        df["NumOfProducts"] == 2,
        df["NumOfProducts"] == 3,
        df["NumOfProducts"] == 1,
        df["NumOfProducts"] >= 4
    ]
    prod_choices = [30, 20, 10, 5]
    prod_pts = np.select(prod_conditions, prod_choices, default=10)

    # Credit Card points (15 pts)
    cc_pts = df["HasCrCard"] * 15

    # Tenure points (15 pts)
    tenure_pts = (df["Tenure"] / 10.0).clip(upper=1.0) * 15

    df["RSI"] = active_pts + prod_pts + cc_pts + tenure_pts
    return df

def calculate_kpis(df, balance_threshold=100000.0):
    # 1. Engagement Retention Ratio
    active_churn = df[df["IsActiveMember"] == 1]["Exited"].mean()
    inactive_churn = df[df["IsActiveMember"] == 0]["Exited"].mean()
    engagement_retention_ratio = active_churn / inactive_churn if inactive_churn > 0 else 0.0

    # 2. Product Depth Index
    retained_avg_prod = df[df["Exited"] == 0]["NumOfProducts"].mean()
    churned_avg_prod = df[df["Exited"] == 1]["NumOfProducts"].mean()
    product_depth_index = retained_avg_prod / churned_avg_prod if churned_avg_prod > 0 else 1.0

    # 3. High-Balance Disengagement Rate
    high_bal_df = df[df["Balance"] >= balance_threshold]
    high_bal_disengagement_rate = (high_bal_df["IsActiveMember"] == 0).mean() if len(high_bal_df) > 0 else 0.0
    high_bal_inactive_churn = high_bal_df[high_bal_df["IsActiveMember"] == 0]["Exited"].mean() if len(high_bal_df) > 0 else 0.0

    # 4. Credit Card Stickiness Score
    cc_churn = df[df["HasCrCard"] == 1]["Exited"].mean()
    no_cc_churn = df[df["HasCrCard"] == 0]["Exited"].mean()
    cc_stickiness_score = cc_churn / no_cc_churn if no_cc_churn > 0 else 1.0

    # 5. Relationship Strength Index
    df_rsi = calculate_rsi(df)
    avg_rsi_retained = df_rsi[df_rsi["Exited"] == 0]["RSI"].mean()
    avg_rsi_churned = df_rsi[df_rsi["Exited"] == 1]["RSI"].mean()

    return {
        "Engagement Retention Ratio": engagement_retention_ratio,
        "Product Depth Index": product_depth_index,
        "High-Balance Disengagement Rate": high_bal_disengagement_rate,
        "High-Balance Inactive Churn Rate": high_bal_inactive_churn,
        "Credit Card Stickiness Score": cc_stickiness_score,
        "Retained Avg RSI": avg_rsi_retained,
        "Churned Avg RSI": avg_rsi_churned
    }

## Step 4: Run Analysis & Print Metrics
Calculate the KPI values and display the profiles statistics.

In [4]:
print("=== CORE KEY PERFORMANCE INDICATORS ===")
kpis = calculate_kpis(df)
for kpi, val in kpis.items():
    print(f"{kpi}: {val:.4f}")

print("\n=== ENGAGEMENT PROFILE BREAKDOWN ===")
df_classified = classify_engagement(df)
profile_stats = df_classified.groupby("EngagementProfile").agg(
    TotalCustomers=("CustomerId", "count"),
    ChurnedCustomers=("Exited", "sum"),
    ChurnRate=("Exited", "mean"),
    AverageBalance=("Balance", "mean"),
    AverageSalary=("EstimatedSalary", "mean")
).reset_index()
print(profile_stats.to_string(index=False))

print("\n=== PRODUCT DEPTH PARADOX ===")
prod_stats = df.groupby("NumOfProducts").agg(
    TotalCustomers=("CustomerId", "count"),
    ChurnedCustomers=("Exited", "sum"),
    ChurnRate=("Exited", "mean")
).reset_index()
print(prod_stats.to_string(index=False))

=== CORE KEY PERFORMANCE INDICATORS ===
Engagement Retention Ratio: 0.5314
Product Depth Index: 1.0468
High-Balance Disengagement Rate: 0.4909
High-Balance Inactive Churn Rate: 0.3277
Credit Card Stickiness Score: 0.9697
Retained Avg RSI: 61.0517
Churned Avg RSI: 46.6676

=== ENGAGEMENT PROFILE BREAKDOWN ===
    EngagementProfile  TotalCustomers  ChurnedCustomers  ChurnRate  AverageBalance  AverageSalary
       Active Engaged            2588               250   0.096600    53071.261190  100753.151016
   Active Low-Product            2563               485   0.189231    98902.019317   98140.098513
  Inactive Disengaged            2493               530   0.212595    24773.029559  100334.013722
Inactive High-Balance            2356               772   0.327674   132540.505399  101225.583735

=== PRODUCT DEPTH PARADOX ===
 NumOfProducts  TotalCustomers  ChurnedCustomers  ChurnRate
             1            5084              1409   0.277144
             2            4590               348 

## Step 5: Visualizing Key Findings
Plot the product paradox and engagement profiles using Plotly.

In [5]:
import plotly.express as px

# Plot Churn Rate by Product Count
fig_prod = px.bar(
    prod_stats,
    x=prod_stats["NumOfProducts"].astype(str),
    y="ChurnRate",
    text=prod_stats["ChurnRate"].apply(lambda x: f"{x*100:.1f}%"),
    title="Product Depth Paradox: Churn Rate by Product Count",
    labels={"x": "Number of Products", "ChurnRate": "Churn Rate"},
    color_discrete_sequence=["#ef4444"]
)
fig_prod.show()

# Plot Churn Rate by Engagement Profile
fig_profile = px.bar(
    profile_stats,
    x="EngagementProfile",
    y="ChurnRate",
    color="EngagementProfile",
    text=profile_stats["ChurnRate"].apply(lambda x: f"{x*100:.1f}%"),
    title="Churn Rate by Customer Engagement Profile",
    color_discrete_map={
        "Active Engaged": "#10b981",
        "Active Low-Product": "#3b82f6",
        "Inactive Disengaged": "#f59e0b",
        "Inactive High-Balance": "#ef4444"
    }
)
fig_profile.show()

## Step 6: Streamlit Dashboard Code Generation
We write out the Streamlit dashboard script `app.py` directly from the notebook.

In [6]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os

st.set_page_config(page_title="Customer Retention Dashboard", layout="wide")

# Custom Styles
st.markdown("""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;600;700&display=swap');
    html, body, [class*="css"] { font-family: 'Outfit', sans-serif; }
    .stApp { background: linear-gradient(135deg, #0e1117 0%, #161a24 100%); color: #f0f2f6; }
    .header-banner { background: linear-gradient(90deg, #1f4068 0%, #162447 50%, #0f1b29 100%); padding: 2rem; border-radius: 16px; border: 1px solid rgba(255,255,255,0.05); text-align: center; margin-bottom: 2rem; }
    .header-banner h1 { font-size: 2.3rem; font-weight: 700; color: white; margin: 0; }
    .header-banner p { font-size: 1.1rem; color: #b0c4de; margin: 0.5rem 0 0 0; }
    .kpi-container { display: flex; gap: 1rem; flex-wrap: wrap; margin-bottom: 2rem; }
    .kpi-card { flex: 1 1 200px; background: rgba(30, 41, 59, 0.45); border: 1px solid rgba(255, 255, 255, 0.08); border-radius: 14px; padding: 1.5rem; text-align: center; }
    .kpi-value { font-size: 2.1rem; font-weight: 700; color: #6366f1; }
    .kpi-title { font-size: 0.9rem; text-transform: uppercase; color: #94a3b8; font-weight: 600; }
    </style>
""", unsafe_allow_html=True)

file_name = "European_Bank.csv"
if not os.path.exists(file_name):
    st.error("Dataset not found!")
    st.stop()

df = pd.read_csv(file_name)

# Sidebar Filters
st.sidebar.markdown("### Filter Controls")
geographies = df["Geography"].unique().tolist()
selected_geos = st.sidebar.multiselect("Geography", options=geographies, default=geographies)
balance_threshold = st.sidebar.number_input("Balance Threshold (€)", value=100000.0)
salary_threshold = st.sidebar.number_input("Salary Threshold (€)", value=120000.0)

filtered_df = df[df["Geography"].isin(selected_geos)]

# KPI calculations
active_churn = filtered_df[filtered_df["IsActiveMember"] == 1]["Exited"].mean()
inactive_churn = filtered_df[filtered_df["IsActiveMember"] == 0]["Exited"].mean()
ratio = active_churn / inactive_churn if inactive_churn > 0 else 0

st.markdown("""
    <div class='header-banner'>
        <h1>European Bank Retention Dashboard</h1>
        <p>Live Analytics for Customer Retention & Engagement Strategy</p>
    </div>
""", unsafe_allow_html=True)

st.markdown(f"""
    <div class='kpi-container'>
        <div class='kpi-card'>
            <div class='kpi-title'>Engagement Retention Ratio</div>
            <div class='kpi-value'>{ratio:.3f}</div>
        </div>
        <div class='kpi-card'>
            <div class='kpi-title'>Active Member Churn</div>
            <div class='kpi-value'>{active_churn*100:.2f}%</div>
        </div>
        <div class='kpi-card'>
            <div class='kpi-title'>Inactive Member Churn</div>
            <div class='kpi-value'>{inactive_churn*100:.2f}%</div>
        </div>
    </div>
""", unsafe_allow_html=True)

tab1, tab2, tab3 = st.tabs(["📈 Churn & Product Utilization", "⚠️ At-Risk Premium Detector", "🧮 RSI Calculator"])

with tab1:
    col1, col2 = st.columns(2)
    with col1:
        prod_stats = filtered_df.groupby("NumOfProducts")["Exited"].mean().reset_index()
        fig = px.bar(prod_stats, x="NumOfProducts", y="Exited", text=prod_stats["Exited"].apply(lambda x: f'{x*100:.1f}%'), title="Churn Rate by Product Count", color_discrete_sequence=['#ef4444'])
        st.plotly_chart(fig, use_container_width=True)
    with col2:
        act_stats = filtered_df.groupby("IsActiveMember")["Exited"].mean().reset_index()
        act_stats["Status"] = act_stats["IsActiveMember"].map({1: "Active", 0: "Inactive"})
        fig2 = px.bar(act_stats, x="Status", y="Exited", text=act_stats["Exited"].apply(lambda x: f'{x*100:.1f}%'), title="Churn Rate by Activity Status", color_discrete_sequence=['#10b981'])
        st.plotly_chart(fig2, use_container_width=True)

with tab2:
    at_risk = filtered_df[
        ((filtered_df["Balance"] >= balance_threshold) | (filtered_df["EstimatedSalary"] >= salary_threshold)) &
        (filtered_df["IsActiveMember"] == 0) &
        (filtered_df["CreditScore"] >= 600)
    ]
    st.metric("At-Risk Premium Customers Count", len(at_risk))
    st.dataframe(at_risk[["CustomerId", "Surname", "CreditScore", "Balance", "EstimatedSalary", "NumOfProducts"]])

with tab3:
    st.markdown("#### Relationship Strength Index")
    calc_active = st.checkbox("Is Active Member", value=True)
    calc_products = st.slider("Product Count", 1, 4, 2)
    calc_has_card = st.checkbox("Has Credit Card", value=True)
    calc_tenure = st.slider("Tenure (Years)", 0, 10, 5)

    score = (40 if calc_active else 0) + (30 if calc_products==2 else (20 if calc_products==3 else (10 if calc_products==1 else 5))) + (15 if calc_has_card else 0) + ((calc_tenure/10)*15)
    st.write(f"### Calculated RSI Score: {score:.1f} / 100")


Writing app.py


## Step 7: Launch Dashboard and Expose to Public URL
Execute the cell below. It will output a **Tunnel IP Key** (which you must copy) and a link. Click the link, enter the tunnel IP key in the bypass screen, and your Streamlit app will load!

In [ ]:
# 1. Fetch the bypass IP key for localtunnel
import urllib.request
print("IP BYPASS KEY (Copy this!):")
print(urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
print("\nStarting Streamlit & Tunnel...")

# 2. Run streamlit in the background and expose it using localtunnel on port 8501
!streamlit run app.py & npx localtunnel --port 8501

IP BYPASS KEY (Copy this!):
34.143.234.240

Starting Streamlit & Tunnel...
⠙⠹⠸⠼⠴

⠦⠧your url is: https://legal-eyes-do.loca.lt
2026-06-06 07:58:25.303 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.143.234.240:8501

2026-06-06 07:59:15.110 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-06-06 07:59:15.142 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-06-06 08:00:26.071 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_conta